In [3]:
from dataclasses import dataclass

class _Unset:
    pass

@dataclass
class BaseContract:
    def to_kwargs(self, *args) -> dict:
        return {key: value for key, value in self.__dict__.items() if (not isinstance(value, _Unset) and key in args)}
    
    def to_named_kwargs(self, **kwargs) -> dict:
        return {kwargs[key]: value for key, value in self.__dict__.items() if (not isinstance(value, _Unset) and key in kwargs)}

@dataclass
class Contract1(BaseContract):
    A: int | None | _Unset = _Unset()
    B: int | None | _Unset = _Unset()
    C: int | None | _Unset = _Unset()


def print_kwargs(**kwargs):
    for key, value in kwargs.items():
        print(f"{key}: {value}")

def servicefunction(contract: Contract1):
    print_kwargs(**contract.to_kwargs('A', 'B'))

def servicefunction_named(contract: Contract1):
    print_kwargs(**contract.to_named_kwargs(A='param_a', B='param_b'))

servicefunction(Contract1(C=3))


In [7]:
from datetime import datetime
from zoneinfo import ZoneInfo

now = datetime.now(ZoneInfo("Europe/Berlin"))
print(now.tzinfo)

Europe/Berlin


In [9]:
from tortoise import fields, run_async
from tortoise.contrib.test import init_memory_sqlite
from tortoise.models import Model
from tortoise.contrib.pydantic import pydantic_model_creator


class Event(Model):
    id = fields.IntField(primary_key=True)
    name = fields.TextField()
    datetime = fields.DatetimeField(null=True)
    username = fields.CharField(max_length=150, unique=True, db_index=True)

    class Meta:
        table = "event"

    def __str__(self):
        return self.name

EventSchema = pydantic_model_creator(Event)


@init_memory_sqlite
async def run() -> None:
    event = await Event.create(name="Test", username="test")
    print(await EventSchema.from_tortoise_orm(event))


if __name__ == "__main__":
    run_async(run())

id=1 name='Test' datetime=None username='test'
